<a href="https://colab.research.google.com/github/pereram/Research_with_SmolLM-/blob/main/Copy_of_Research_with_SmolLm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import math
import pickle
from datasets import load_dataset
import regex
from tqdm import tqdm
import os
import json
import time
import numpy as np
from collections import defaultdict
from transformers import AutoTokenizer

In [ ]:
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [ ]:
num_documents=500 # Define the number of documents to load
# Print a loading message
print(f"Loading HuggingFaceTB/smollm-corpus dataset (cosmopedia-v2)...")
# Load a pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M", token=False)

# --------  Load the dataset in streaming mode. ----------------
# Cosmopedia v2 is an enhanced version of Cosmopedia, the largest synthetic
#dataset for pre-training, consisting of over 39 million textbooks, blog posts, and stories
# The foillowing line of code is taken from huggingface HuggingFaceTB/smollm-corpus page.

dataset = load_dataset("HuggingFaceTB/smollm-corpus", "cosmopedia-v2", split="train", streaming=True, token=False)



texts = [] # Initialize an empty list to store the loaded texts
for i, item in enumerate(dataset): # Iterate through the dataset items
    if i >= num_documents: # Check if the desired number of documents has been loaded
        break # Exit the loop if enough documents are loaded
    texts.append(item["text"]) # Add the 'text' field of the current item to the texts list

print(f"Loaded {len(texts)} documents") # Print the number of documents loaded
total_tokens = sum(len(tokenizer.encode(text)) for text in texts) # Calculate the total number of tokens across all documents
print(f"Total tokens: {total_tokens:,}") # Print the total number of tokens
words_cnt = sum(len(text.split()) for text in texts)
print(f"Total words:{words_cnt:,}")
print(f"Average tokens per document: {total_tokens / len(texts):.1f}") # Print the average number of tokens per document

#--- printing above  numbers ------
# --------Advanced Formatting Techniques ----
'''
---F-Strings (Recommended): ----
  Use a prefix f and curly braces {} to embed variables.
  You can specify fixed-point precision for floats using : .nf.

  pi = 3.14159
  print(f"Pi to 2 decimal places is {pi:.2f}")  # Output: Pi to 2 decimal places is 3.14


--- Thousands Separators: --------
  Use the :, specifier within an f-string to make large numbers more readable.

  print(f"Total: {1000000:,}")  # Output: Total: 1,000,000


'''



Loading HuggingFaceTB/smollm-corpus dataset (cosmopedia-v2)...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded 500 documents
Total tokens: 366,493
Total words:277,221
Average tokens per document: 733.0


'\n---F-Strings (Recommended): ----\n  Use a prefix f and curly braces {} to embed variables.\n  You can specify fixed-point precision for floats using : .nf.\n\n  pi = 3.14159\n  print(f"Pi to 2 decimal places is {pi:.2f}")  # Output: Pi to 2 decimal places is 3.14\n\n\n--- Thousands Separators: --------\n  Use the :, specifier within an f-string to make large numbers more readable.\n\n  print(f"Total: {1000000:,}")  # Output: Total: 1,000,000\n\n\n'